# Day 3 - GroupBy and Aggregations (The Shuffles)This notebook covers aggregation operations in PySpark:- Reading Parquet files- Basic aggregations (without GroupBy)- GroupBy with aggregations- Understanding shuffle operations

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .appName("spark_day3")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/05 13:10:29 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/05 13:10:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/biswa/practice/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/05 13:10:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Load Parquet DataRead the parquet data saved in Day 1. Parquet preserves schema and is efficient for analytics.

In [5]:
parquet_path = r"../dataset/parquet_output"
df = spark.read.parquet(parquet_path)

In [6]:
from pyspark.sql import functions as F

## 3. Basic Aggregations (without GroupBy)Aggregations without `groupBy()` compute over the entire DataFrame, returning a single row.

In [19]:
df_1 = df.select(
    F.count("ID").alias("total_customers"),
    F.min("Score").alias("min_score"),
    F.avg("Score").alias("avg_score"),
    F.max("Score").alias("max_score")
)
df_1.show()

+---------------+---------+---------+---------+
|total_customers|min_score|avg_score|max_score|
+---------------+---------+---------+---------+
|              5|      350|    625.0|      900|
+---------------+---------+---------+---------+



## 4. GroupBy with Aggregations`groupBy()` groups rows by column(s), then `.agg()` applies aggregation functions per group.**Triggers a Shuffle** - data is redistributed across partitions by group key.

In [17]:
df_2 = df.groupBy("Country").agg(
    F.count("ID").alias("total_customers"),
    F.min("Score").alias("min_score"),
    F.avg("Score").alias("avg_score"),
    F.max("Score").alias("max_score")
)
df_2.show()

+-------+---------------+---------+---------+---------+
|Country|total_customers|min_score|avg_score|max_score|
+-------+---------------+---------+---------+---------+
|Germany|              2|      350|    425.0|      500|
|    USA|              3|      750|    825.0|      900|
+-------+---------------+---------+---------+---------+



## 5. Understanding Shuffle Operations### What is a Shuffle?A shuffle is the process of redistributing data across partitions so that rows with the same key end up in the same partition. This happens during:- `groupBy()` operations- `join()` operations- `repartition()` / `coalesce()`- `orderBy()` / `sort()`### Shuffle Phases:1. **Map Phase** - Each partition processes its data, writes to local disk2. **Shuffle Phase** - Data is transferred across network to target partitions3. **Reduce Phase** - Target partitions read and aggregate data### Performance Impact:- **Network I/O** - Data moves between executors- **Disk I/O** - Spill to disk if memory insufficient- **Serialization** - Data must be serialized for transfer### Optimization Tips:- Filter early (before groupBy)- Use `coalesce()` instead of `repartition()` when reducing partitions- Consider broadcast joins for small DataFrames- Tune `spark.sql.shuffle.partitions` (default 200)

In [20]:
spark.stop()